# Dataset de entrenamiento — cross-encoder: ampliación del corpus vía Corte Constitucional (embudo aleatorio)

Amplía el corpus de la parte 1 (21 sentencias curadas de redal.org, notebook `ds_parte1_redal.ipynb`)
usando el índice completo de sentencias de la Corte Constitucional (Tutela y Unificación) vía la
API de Datos Abiertos. Aplica un embudo de tres pasos para no gastar tokens de LLM en sentencias
irrelevantes:

1. Índice completo (API pública, sin scraping del buscador JS de la Corte)
2. Prefiltro barato de keywords sobre el texto completo
3. Filtro de pertinencia barato vía LLM (Haiku, solo el inicio del texto)
4. Extracción completa vía LLM (Sonnet, texto completo) solo para lo que pasó los filtros anteriores

Diseñado para correr primero un **piloto chico** (parametrizable, `SAMPLE_SIZE`) y validar que
el pipeline funciona de punta a punta antes de lanzar una muestra grande — la API de Anthropic
tiene cuota limitada. Cada corrida **acumula** sobre la anterior: los resultados (pares y
descartes) se guardan en un único CSV por tipo que se recarga al inicio, y las sentencias que
ya aparecen ahí (aceptadas o descartadas) se excluyen del muestreo — subir `SAMPLE_SIZE` y
volver a correr nunca reprocesa una sentencia ya explorada.

Corre en paralelo a `ds_parte3_research_list.ipynb` (candidatas curadas por research dirigido en
vez de muestreo aleatorio) — ambos comparten la misma blacklist (`dataset_cross_encoder.csv` /
`descartados.csv`, el mismo archivo consolidado que también incluye las sentencias de
`ds_parte1_redal.ipynb`), así que ninguno reprocesa lo que otro ya vio.

**Contenido:**
1. Setup
2. Índice de sentencias (Datos Abiertos)
3. Muestra incremental (excluye sentencias ya procesadas)
4. Descarga y limpieza de texto completo
5. Prefiltro de keywords
6a. Filtro barato de pertinencia (Haiku)
6b. Extracción estructurada completa (Sonnet, solo pertinentes)
7. Construcción de pares (mismo criterio que la parte 1)
8. Loop principal con checkpointing
9. Resultado acumulado

## 1. Setup

In [1]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q beautifulsoup4 requests pandas anthropic python-dotenv

import os
import re
import time
import json

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import anthropic

load_dotenv()  # lee ANTHROPIC_API_KEY del .env en la raíz del repo
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

HEADERS = {"User-Agent": "Mozilla/5.0"}

# Rutas acumuladas — cada corrida se suma a lo que ya hay, nunca se sobreescribe desde cero.
# dataset_cross_encoder.csv es el archivo consolidado (redal + corte_const + research).
PARES_PATH = "../../data/dataset_cross_encoder.csv"
DESCARTES_PATH = "../../data/descartados.csv"
COLUMNAS_PARES = ["consulta", "articulo", "tipo", "label", "sentencia_origen"]
COLUMNAS_DESCARTES = ["sentencia", "razon"]

def cargar_si_existe(path, columnas):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=columnas)

print("Setup listo.")

Setup listo.


## 2. Índice de sentencias (Datos Abiertos)

API pública de Datos Abiertos (Socrata) con el índice completo de sentencias de la Corte
Constitucional — no requiere scraping del buscador JS de la Corte. Se filtra a **T** (Tutela)
y **SU** (Unificación), excluyendo **C** (control de constitucionalidad en abstracto — igual
que en la parte 1, no aportan una consulta de un caso real). El índice completo (`~29.400`
sentencias, `~21.900` T/SU) no trae texto ni tema — solo metadata (número, tipo, fecha,
magistrado) — el texto se descarga aparte en la sección 4.

In [2]:
INDICE_URL = "https://www.datos.gov.co/resource/v2k4-2t8s.json"

# Parametrizable: dejar en None para no filtrar por fecha
FECHA_DESDE = None  # ej. "2000-01-01"
FECHA_HASTA = None  # ej. "2025-12-31"

def construir_where():
    condiciones = ["sentencia_tipo in ('T','SU')"]
    if FECHA_DESDE:
        condiciones.append(f"fecha_sentencia >= '{FECHA_DESDE}'")
    if FECHA_HASTA:
        condiciones.append(f"fecha_sentencia <= '{FECHA_HASTA}'")
    return " AND ".join(condiciones)

def descargar_indice_completo(page_size=5000):
    """Pagina sobre la API de Datos Abiertos hasta traer el índice completo T/SU."""
    filas = []
    offset = 0
    where = construir_where()
    while True:
        params = {"$where": where, "$limit": page_size, "$offset": offset, "$order": ":id"}
        resp = requests.get(INDICE_URL, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        pagina = resp.json()
        if not pagina:
            break
        filas.extend(pagina)
        offset += page_size
        if len(pagina) < page_size:
            break
    return pd.DataFrame(filas)

df_indice = descargar_indice_completo()
print(f"Sentencias T/SU en el índice: {len(df_indice)}")
df_indice[["sentencia", "sentencia_tipo", "fecha_sentencia"]].head()

Sentencias T/SU en el índice: 21872


,sentencia,sentencia_tipo,fecha_sentencia
0,T-012/92,T,1992-02-25T00:00:00.000
1,T-001/92,T,1992-04-03T00:00:00.000
2,T-002/92,T,1992-05-08T00:00:00.000
3,T-003/92,T,1992-05-11T00:00:00.000
4,T-006/92,T,1992-05-12T00:00:00.000


## 3. Muestra incremental (excluye sentencias ya procesadas)

Se cargan los CSV acumulados de corridas anteriores (si existen) y se arma el conjunto de
sentencias ya exploradas — tanto las que terminaron en pares (`sentencia_origen` en
`dataset_cross_encoder.csv`, el archivo consolidado) como las descartadas en cualquier paso del
embudo (`sentencia` en `descartados.csv`). Esa "blacklist" se resta del índice antes de
muestrear, así que subir `SAMPLE_SIZE` y volver a correr esta celda en adelante nunca vuelve a
gastar una llamada a la API en algo ya visto.

`SAMPLE_SIZE` es la cantidad de sentencias **nuevas** a intentar en esta corrida (no el total
acumulado). Se corre primero chico para validar el pipeline de punta a punta (la mayoría del
índice T no es de derecho laboral individual — el rendimiento esperado tras el embudo completo
es bajo en una muestra chica, eso es normal); si corre sin errores y el criterio de pertinencia
se ve razonable, subir a una muestra grande (ej. 500).

In [3]:
SAMPLE_SIZE = 500  # cantidad de sentencias NUEVAS a intentar en esta corrida
RANDOM_STATE = 42  # fija la semilla para que cada corrida sea reproducible

df_pares_acum = cargar_si_existe(PARES_PATH, COLUMNAS_PARES)
df_descartes_acum = cargar_si_existe(DESCARTES_PATH, COLUMNAS_DESCARTES)

ya_procesadas = set(df_pares_acum["sentencia_origen"]) | set(df_descartes_acum["sentencia"])
print(f"Ya procesadas en corridas anteriores: {len(ya_procesadas)}")

df_candidatas = df_indice[~df_indice["sentencia"].isin(ya_procesadas)]
print(f"Candidatas nuevas disponibles: {len(df_candidatas)}")

df_muestra = df_candidatas.sample(n=min(SAMPLE_SIZE, len(df_candidatas)), random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Muestra de esta corrida: {len(df_muestra)} sentencias")
df_muestra["sentencia_tipo"].value_counts()

Ya procesadas en corridas anteriores: 947
Candidatas nuevas disponibles: 20927
Muestra de esta corrida: 500 sentencias


sentencia_tipo
T     488
SU     12
Name: count, dtype: int64

## 4. Descarga y limpieza de texto completo

Patrón de URL de la relatoría de la Corte Constitucional:
`https://www.corteconstitucional.gov.co/relatoria/{año}/{TIPO}{sep}{numero}-{año corto}.htm`.

**Importante, verificado manualmente contra el sitio real:**

- El separador tras el tipo es `-` para `T` y `C` (`T-528-17.htm`) pero **no existe para `SU`**
  (`SU380-21.htm`, no `SU-380-21.htm`) — con guion, la URL resuelve a una página vacía de la
  app Angular del sitio, no a un 404 explícito, así que es un error silencioso si no se
  contempla. Esa página vacía se detecta de forma confiable por la presencia del tag
  `<app-root` en el HTML crudo — **no por tamaño**: se probó primero con un umbral de bytes y
  descartó por error sentencias reales pero cortas (ej. `T-420/99`, 15 KB de contenido legítimo,
  por debajo de un umbral de 20 KB elegido a ojo).
- El HTML declara charset `windows-1252`, no UTF-8 — decodificarlo mal genera acentos
  corruptos.
- El ancla de inicio varía: sentencias de ~1992 dicen *"Sentencia No. T-012/92"* en vez de
  *"Sentencia T-012/92"* — se acepta el `No.` opcional.
- El cierre del fallo **no sigue una fórmula fija**: unas terminan en *"Notifíquese,
  comuníquese, publíquese y cúmplase."*, otras en *"Comuníquese y cúmplase."*, y varias no
  contienen ninguna de esas frases (ej. solo *"LIBRAR la comunicación a que se refiere el
  artículo 36..."* sin mencionar "cúmplase"). Por eso el ancla de cierre es **best-effort**: si
  se encuentra, recorta ruido de aclaraciones de voto/pie de página; si no se encuentra, se
  conserva el texto completo desde el inicio en vez de descartar la sentencia — el truncado a
  `max_chars` en la sección 6 ya acota lo que se le manda al LLM.
- Algunos números de radicado individuales no resuelven (huecos normales en la numeración) —
  se descartan como cualquier otro fallo de descarga real.

In [4]:
def construir_url_relatoria(tipo, sentencia):
    # sentencia viene como "T-528/17" o "SU-380/21" — separa número y año corto
    numero, anio_corto = sentencia.split("-", 1)[1].split("/")
    anio_largo = ("20" if int(anio_corto) <= 50 else "19") + anio_corto
    sep = "" if tipo == "SU" else "-"
    return f"https://www.corteconstitucional.gov.co/relatoria/{anio_largo}/{tipo}{sep}{numero}-{anio_corto}.htm"

RE_INICIO = re.compile(r"Sentencia\s+(?:No\.?\s*)?(?:T|SU|C)\s*-?\s*\d+\s*/\s*\d+", re.IGNORECASE)
RE_FIN = re.compile(r"comuníquese.{0,80}?cúmplase\.?", re.IGNORECASE)

def descargar_texto_sentencia(url):
    """Descarga y limpia el texto de una sentencia. Devuelve None si falla o si
    la página no trae contenido real (shell de la app Angular del sitio)."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        if b"<app-root" in resp.content:
            return None  # shell vacío de la SPA del sitio, sin contenido real
        soup = BeautifulSoup(resp.content, "html.parser", from_encoding="windows-1252")
        texto = re.sub(r"\s+", " ", soup.get_text(" ")).strip()

        m_inicio = RE_INICIO.search(texto)
        if not m_inicio:
            return None  # no se encontró el título de la sentencia, contenido sospechoso
        texto = texto[m_inicio.start():]

        m_fin = RE_FIN.search(texto)
        if m_fin:
            texto = texto[:m_fin.end()]  # recorta aclaraciones de voto / pie si se encontró el cierre

        return texto
    except Exception:
        return None

## 5. Prefiltro de keywords

Filtro barato (sin LLM) sobre el texto ya limpio: solo pasan al filtro de pertinencia real las
sentencias que mencionan al menos uno de estos términos de alcance del dominio.

**Lista ampliada tras revisar el alcance real del corpus de la parte 1** (21 sentencias de
redal.org, ya aceptado como válido): la lista original solo cubría contratación/jornada/despido,
pero el corpus real de la parte 1 incluye pagos y descuentos salariales, licencias,
discriminación laboral y debido proceso disciplinario — ninguno de esos términos estaba en la
lista original, así que se habrían perdido casos equivalentes en la parte 2 sin que el LLM
llegara siquiera a evaluarlos.

In [5]:
KEYWORDS_ALCANCE = [
    "estabilidad laboral reforzada", "contrato de trabajo", "despido",
    "terminación del contrato", "jornada laboral", "prestaciones sociales",
    "presunción de relación laboral", "liquidación", "recargo nocturno",
    "recargo dominical", "reintegro laboral",
    # ampliados: cubren temas presentes en el corpus real de la parte 1 que la lista
    # original no capturaba (pagos/descuentos, licencias, discriminación, disciplinario)
    "relación laboral", "contrato realidad", "justa causa", "indemnización",
    "salario", "cesantías", "vacaciones", "ius variandi", "período de prueba",
    "discriminación laboral", "licencia de maternidad", "fuero de maternidad",
    "proceso disciplinario", "descuento salarial", "acoso laboral",
]

def pasa_prefiltro_keywords(texto):
    texto_lower = texto.lower()
    return any(kw in texto_lower for kw in KEYWORDS_ALCANCE)

## 6a. Filtro barato de pertinencia (Claude Haiku)

De las sentencias que pasan el prefiltro de keywords, la gran mayoría termina de todas formas
descartada por no ser pertinente (en la corrida de 200, 114 de 125 — el 91%). Hacer la
extracción completa (Sonnet, texto completo, hasta 1200 tokens de salida) para decidir
pertinencia desperdicia la mayor parte de ese costo en sentencias que se iban a descartar
igual. Este filtro barato usa **Claude Haiku** sobre solo los primeros ~2.500 caracteres (donde
ya suele quedar claro el tema del caso) y responde nada más `pertinente_alcance` + una razón
breve — la extracción completa (sección 6b) corre únicamente para las que pasan aquí.

**Criterio de exclusión** (afinado auditando el dataset acumulado):

1. **Pensión es exclusión incondicional** — jubilación, sustitución pensional o bono
   pensional, `false` sin excepción, sin importar qué tan laboral suene el resto del caso
   (ej. `T-398/15`).
2. **Empleado público vs. trabajador oficial** — Colombia distingue el "empleado público"
   (régimen estatutario/administrativo: docentes oficiales del magisterio, funcionarios de
   carrera) que NO aplica, del "trabajador oficial" (empresas industriales/comerciales del
   Estado, regido por normas tipo CST) que SÍ puede aplicar (ej. `T-008/15`).
3. **Seguridad social en salud (EPS) y trabajadores independientes** — un trámite contra una
   EPS o un reclamo sin relación de empleador no es un litigio laboral (ej. `T-943/07`).
4. **Estudiantes/practicantes en formación** bajo convenio docencia-servicio, sin contrato de
   trabajo (ej. `T-948/08`).

El fragmento que evalúa el gate arranca en `ANTECEDENTES` cuando ese encabezado existe (ahí
están los hechos reales del caso — los primeros ~2.500 caracteres a veces son preámbulo
doctrinal sin el dato clave) y cae de vuelta al inicio del texto si no aparece.

In [6]:
PROMPT_GATE = """Lee este fragmento del inicio de una sentencia de la Corte Constitucional colombiana (puede estar incompleto, es solo el comienzo). Con base en el título/tema del caso, decide si es relevante para un dataset de derecho laboral individual colombiano.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"pertinente_alcance": true o false, "razon": "una frase breve"}}

pertinente_alcance=true SOLO si el caso trata sobre una relación laboral INDIVIDUAL regida por el Código Sustantivo del Trabajo (o normas equivalentes para trabajadores oficiales): contratación, jornada, terminación del contrato, salario y pagos (constitutivos o no de salario, descuentos), licencias, vacaciones, cesantías (bajo régimen CST/Ley 50 de 1990), traslados (ius variandi), procedimiento disciplinario del empleador, estabilidad laboral reforzada, discriminación laboral en el empleo.

pertinente_alcance=false SIEMPRE, sin excepción, si:
- El tema central es PENSIÓN o SEGURIDAD SOCIAL en cualquier forma: pensión de vejez, invalidez, sobrevivientes, jubilación, sustitución pensional, bono pensional, régimen de prima media o ahorro individual. Esto aplica sin importar qué tan laboral suene el resto del caso — pensión siempre es false.
- El tema central es seguridad social EN SALUD (EPS): pago de licencias (maternidad, incapacidad) reclamado contra una EPS, cobertura o negación de servicios de salud, afiliación al sistema de salud. Es un trámite de seguridad social en salud, no un litigio laboral entre empleador y trabajador — false aunque mencione licencia de maternidad u otro tema nominalmente laboral.
- El accionante es un TRABAJADOR INDEPENDIENTE (por cuenta propia, sin empleador) — el CST no rige relaciones sin empleador, así que no hay relación laboral individual que evaluar. False.
- El accionante es un ESTUDIANTE o PRACTICANTE en formación (práctica académica, convenio docencia-servicio) sin contrato de trabajo real — no hay relación laboral, aunque el conflicto sea con la institución donde hace la práctica. False.
- El empleador es una entidad estatal Y el vínculo es de "empleado público" bajo régimen estatutario/administrativo (docente oficial del magisterio, funcionario de carrera administrativa, personal nombrado, régimen especial propio como el Fondo de Prestaciones Sociales del Magisterio) — NO CST. Si el fundamento del caso son estatutos especiales (ej. Ley 91 de 1989, Estatuto Docente, Decreto 1042 de 1978, normas de carrera administrativa) en vez de CST/Ley 50 de 1990, es false. Excepción: si el vínculo es de "trabajador oficial" (típico de empresas industriales y comerciales del Estado, regido por normas tipo CST o convención colectiva), sí puede ser pertinente.
- Es derecho colectivo (sindicatos, negociación colectiva, huelga, fuero sindical).
- Es función pública en cualquier otro sentido, o el tema es ajeno al laboral individual.
- Lo laboral aparece solo de forma incidental, sin ser el objeto central de la decisión.

Si el fragmento no deja claro el tema o el tipo de vínculo laboral, responde false.

Fragmento:
{texto}
"""

RE_ANTECEDENTES = re.compile(r"ANTECEDENTES")

def filtro_pertinencia_barato(texto, numero, chars_gate=2500):
    m = RE_ANTECEDENTES.search(texto)
    inicio = m.start() if m else 0
    fragmento = texto[inicio:inicio + chars_gate]
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{"role": "user", "content": PROMPT_GATE.format(texto=fragmento)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error en filtro de pertinencia {numero}: {e}")
        return None

## 6b. Extracción estructurada completa (Claude Sonnet, solo pertinentes)

Igual al prompt de extracción de la parte 1 — ya no necesita el campo `pertinente_alcance`
porque eso ya se decidió en la sección 6a; esta celda solo corre para sentencias que ya
pasaron ese filtro.

In [7]:
PROMPT_TEMPLATE = """Lee la siguiente sentencia laboral colombiana. Extrae exactamente esta información y devuelve SOLO un JSON válido, sin texto adicional ni backticks:

{{
  "hechos_resumidos": "los hechos del caso en 2-3 líneas, en lenguaje coloquial, como si un trabajador lo contara (ej. 'me despidieron después de...' o 'trabajé X años y...')",
  "pretension": "qué pedía el demandante, en pocas palabras",
  "articulos_fundamento_directo": ["artículos cuya interpretación/aplicación fue DETERMINANTE para resolver el punto concreto en disputa de este caso. Pregunta guía: si se quitara este artículo, ¿cambiaría el razonamiento de por qué se concedió o negó la pretensión específica? Si sí, va aquí. Formato 'CST Art. X' o 'Ley X de YYYY, Art. Y'"],
  "articulos_marco_general": ["artículos que la sentencia cita pero que son principios generales, reglas de interpretación/remisión, o contexto normativo (ej. favorabilidad, analogía, primacía de la realidad, normas constitucionales genéricas) — NO decidieron el punto específico del caso, cualquier sentencia laboral podría citarlos. Mismo formato."],
  "decision": "concedida" o "negada" o "parcial",
  "elemento_no_acreditado": "si la pretensión fue negada o parcial, qué elemento/requisito no se acreditó según el juez; si fue concedida totalmente, deja este campo vacío"
}}

Sentencia:
{texto}
"""

def extraer_estructura(texto, numero, max_chars=15000):
    texto_truncado = texto[:max_chars]
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1200,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(texto=texto_truncado)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error extrayendo {numero}: {e}")
        return None

## 7. Construcción de pares (mismo criterio que la parte 1)

Idéntico a la parte 1: positivo por cada artículo en `articulos_fundamento_directo`, negativo
fácil desde un pool curado de temas laborales distintos, negativo difícil (placeholder) vía
segunda llamada al LLM. Mismo esquema de columnas — ambos CSVs son concatenables sin
transformación adicional.

In [8]:
POOL_NEGATIVOS_FACILES = [
    {"articulo": "CST Art. 236", "tema": "licencia de maternidad"},
    {"articulo": "CST Art. 186", "tema": "vacaciones anuales"},
    {"articulo": "CST Art. 161", "tema": "jornada de trabajo"},
    {"articulo": "Ley 100 de 1993, Art. 13", "tema": "seguridad social"},
]

PROMPT_NEGATIVO_DIFICIL = """Dada esta consulta de un caso laboral colombiano:

{consulta}

Y sabiendo que los artículos correctamente aplicables son: {articulos_correctos}

Dame UN artículo real del derecho laboral colombiano (CST, leyes laborales) que esté relacionado temáticamente con la consulta pero que NO sea el fundamento correcto de la decisión — es decir, un artículo que alguien podría confundir con el correcto pero que no aplica aquí.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"articulo_incorrecto": "...", "por_que_se_confunde": "..."}}
"""

def generar_negativo_dificil(consulta, articulos_correctos):
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": PROMPT_NEGATIVO_DIFICIL.format(
                consulta=consulta, articulos_correctos=articulos_correctos)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error generando negativo difícil: {e}")
        return None

def construir_pares(extraccion, numero):
    consulta = extraccion["hechos_resumidos"]
    articulos_correctos = extraccion.get("articulos_fundamento_directo", [])
    articulos_marco = extraccion.get("articulos_marco_general", [])
    ya_citados = set(articulos_correctos) | set(articulos_marco)

    if not articulos_correctos:
        return None

    pares = [
        {"consulta": consulta, "articulo": art, "tipo": "positivo", "label": 1}
        for art in articulos_correctos
    ]

    candidatos_faciles = [n for n in POOL_NEGATIVOS_FACILES if n["articulo"] not in ya_citados]
    if candidatos_faciles:
        pares.append({
            "consulta": consulta,
            "articulo": candidatos_faciles[0]["articulo"],
            "tipo": "negativo_facil",
            "label": 0,
        })

    negativo_dificil = generar_negativo_dificil(consulta, articulos_correctos)
    if negativo_dificil:
        pares.append({
            "consulta": consulta,
            "articulo": negativo_dificil["articulo_incorrecto"],
            "tipo": "negativo_dificil_placeholder",
            "label": 0,
        })

    df = pd.DataFrame(pares)
    df["sentencia_origen"] = numero
    return df

## 8. Loop principal con checkpointing

Recorre la muestra, aplicando el embudo completo por sentencia con manejo de errores
individual (una sentencia que falla no tumba el loop). Guarda checkpoint del CSV de pares y
del log de descartes cada `CHECKPOINT_EVERY` sentencias procesadas, para no perder todo si se
cae la conexión a mitad de una corrida larga — cada checkpoint escribe lo acumulado de
corridas anteriores **más** lo nuevo de esta corrida, nunca solo esta corrida. Pausa entre
requests HTTP y entre llamadas a la API de Anthropic para no golpear ninguno de los dos
servicios en paralelo sin pausas.

In [9]:
CHECKPOINT_EVERY = 10

nuevos_pares = []  # lista de DataFrames, uno por sentencia aceptada en ESTA corrida
nuevos_descartes = []  # {"sentencia": ..., "razon": ...}, de ESTA corrida

def guardar_checkpoint():
    partes = [df_pares_acum] + nuevos_pares
    pd.concat(partes, ignore_index=True).to_csv(PARES_PATH, index=False)
    pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)],
              ignore_index=True).to_csv(DESCARTES_PATH, index=False)

for i, row in df_muestra.iterrows():
    numero = row["sentencia"]
    tipo = row["sentencia_tipo"]
    print(f"[{i+1}/{len(df_muestra)}] Procesando {numero}...")

    try:
        url = construir_url_relatoria(tipo, numero)
        texto = descargar_texto_sentencia(url)
        time.sleep(0.5)

        if texto is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_texto_o_url_no_resuelve"})
            continue

        if not pasa_prefiltro_keywords(texto):
            nuevos_descartes.append({"sentencia": numero, "razon": "no_pasa_prefiltro_keywords"})
            continue

        gate = filtro_pertinencia_barato(texto, numero)
        time.sleep(0.5)

        if gate is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_filtro_pertinencia"})
            continue

        if not gate.get("pertinente_alcance"):
            nuevos_descartes.append({
                "sentencia": numero,
                "razon": f"llm_no_pertinente: {gate.get('razon', '')}",
            })
            continue

        extraccion = extraer_estructura(texto, numero)
        time.sleep(1)

        if extraccion is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_extraccion_llm"})
            continue

        df_par = construir_pares(extraccion, numero)
        time.sleep(1)

        if df_par is not None:
            nuevos_pares.append(df_par)
        else:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_articulos_fundamento_directo"})

    except Exception as e:
        nuevos_descartes.append({"sentencia": numero, "razon": f"error_inesperado: {e}"})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        guardar_checkpoint()
        print(f"  checkpoint guardado ({i+1}/{len(df_muestra)})")

guardar_checkpoint()
print(f"\nProcesamiento completo. Pares generados de {len(nuevos_pares)} sentencias pertinentes nuevas.")
print(f"Descartadas en esta corrida: {len(nuevos_descartes)} de {len(df_muestra)}")

[1/500] Procesando T-354/24...


[2/500] Procesando T-889A/06...


[3/500] Procesando T-320/07...


[4/500] Procesando T-1132/00...


[5/500] Procesando T-195/23...


...

[497/500] Procesando T-139/94...


[498/500] Procesando T-468/99...


[499/500] Procesando T-027/06...


[500/500] Procesando T-212/21...



Procesamiento completo. Pares generados de 19 sentencias pertinentes nuevas.
Descartadas en esta corrida: 481 de 500


## 9. Resultado acumulado

In [10]:
df_dataset_parte2 = pd.concat([df_pares_acum] + nuevos_pares, ignore_index=True) if nuevos_pares else df_pares_acum
df_descartes_total = pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)], ignore_index=True)

print(f"--- Esta corrida ({len(df_muestra)} sentencias nuevas intentadas) ---")
print(f"Pares nuevos: {sum(len(df) for df in nuevos_pares)} de {len(nuevos_pares)} sentencias pertinentes")

print(f"\n--- Acumulado total ---")
print(f"Total de pares: {len(df_dataset_parte2)}")
if not df_dataset_parte2.empty:
    print(f"Sentencias representadas: {df_dataset_parte2['sentencia_origen'].nunique()}")
    print(df_dataset_parte2["tipo"].value_counts())

print("\nRazones de descarte (acumulado):")
if not df_descartes_total.empty:
    print(df_descartes_total["razon"].apply(lambda r: r.split(":")[0]).value_counts())

print(f"\nGuardado en {PARES_PATH}")
print(f"Log de descartes en {DESCARTES_PATH}")
df_dataset_parte2.head()

--- Esta corrida (500 sentencias nuevas intentadas) ---
Pares nuevos: 69 de 19 sentencias pertinentes

--- Acumulado total ---
Total de pares: 498
Sentencias representadas: 125
tipo
positivo                        248
negativo_facil                  125
negativo_dificil_placeholder    125
Name: count, dtype: int64

Razones de descarte (acumulado):
razon
llm_no_pertinente                   772
no_pasa_prefiltro_keywords          453
sin_articulos_fundamento_directo     49
sin_texto_o_url_no_resuelve          45
revision_manual                       5
Name: count, dtype: int64

Guardado en ../../data/dataset_cross_encoder.csv
Log de descartes en ../../data/descartados.csv


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL-3630/22
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL-3630/22
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL-3630/22
3,Trabajé como operador de bus articulado desde ...,CST Art. 130,negativo_dificil_placeholder,0,SL-3630/22
4,Trabajé para el ISS desde septiembre de 2000 h...,"Decreto 2127 de 1945, Art. 20",positivo,1,SL-2858/22
